1. Splittare training set e test set
2. encoding dei dati categorici --> tutti dati numeri
3. Imputazione ( Pulizia del dataset )
4. Feature Selection

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Caricare il file CSV
df = pd.read_csv('./heart_disease_uci.csv', sep=';')
df.drop(columns=['id'],inplace=True)


In [2]:
#1 spittare dataset

X = df.drop(columns=['num'])
y = df.num

#seconda lezione ora stratifichiamo la divisione del dataset in modo da mantenere la stessa proporzione di classi sia nel training set che nel test set.
X_tr, X_te, y_tr, y_te = train_test_split(X,y, test_size=0.2, random_state=42, stratify=df['dataset']) 
'''
X_tr, X_te, y_tr, y_te = train_test_split(
X, y, test_size=0.2, random_state=42)

'''


'\nX_tr, X_te, y_tr, y_te = train_test_split(\nX, y, test_size=0.2, random_state=42)\n\n'

In [3]:

#encoding
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder()
var_cat = ['sex','cp','fbs','restecg','exang','slope','thal','dataset']

X_tr[var_cat] = encoder.fit_transform(X_tr[var_cat])
X_te[var_cat] = encoder.transform(X_te[var_cat])

print(X_tr)


     age  sex  dataset   cp  trestbps   chol  fbs  restecg  thalch  exang  \
328   37  1.0      1.0  0.0     120.0  223.0  0.0      1.0   168.0    0.0   
177   56  1.0      0.0  0.0     132.0  184.0  0.0      0.0   105.0    1.0   
652   56  1.0      2.0  2.0     120.0    0.0  0.0      1.0    97.0    0.0   
626   51  0.0      2.0  0.0     120.0    0.0  NaN      1.0   127.0    1.0   
25    50  0.0      0.0  2.0     120.0  219.0  0.0      1.0   158.0    0.0   
..   ...  ...      ...  ...       ...    ...  ...      ...     ...    ...   
845   76  1.0      3.0  2.0     104.0    NaN  0.0      0.0   120.0    0.0   
404   49  0.0      1.0  1.0     110.0    NaN  0.0      1.0   160.0    0.0   
380   45  1.0      1.0  0.0     140.0  224.0  0.0      1.0   144.0    0.0   
403   48  1.0      1.0  2.0     110.0  211.0  0.0      1.0   138.0    0.0   
510   48  1.0      1.0  0.0     120.0  260.0  0.0      1.0   115.0    0.0   

     oldpeak  slope   ca  thal  
328      0.0    NaN  NaN   1.0  
177      

In [7]:
#imputazione seconda lezione con pandas per stratificare anche l'imputazione
var_cat = ['sex','cp','fbs','restecg','exang','slope','thal','dataset']
col_num = [col for col in X_tr.columns if col not in var_cat]  
print(col_num)

print("Valori mancanti prima dell'imputazione:")

#X_tr.isna().sum() restituisce il numero di valori mancanti per ogni colonna del DataFrame X_tr.
#crea una tabella di vero e falso con True se il valore è mancante e False se non lo è, poi somma i valori True per ogni colonna per ottenere il numero totale di valori mancanti in ciascuna colonna.
print(X_tr.isna().sum())

for dataset in X_tr['dataset'].unique():
    for col in col_num:
        #dati numerici: imputazione con mediana
        #X_tr.loc seleziono le osservazioni di trainset che hanno la feature dataset = alla viaribile dataset e prendo la colonna seleziona col

        median = X_tr.loc[X_tr['dataset'] == dataset, col].median()
        # Assegnazione esplicita invece di inplace=True per evitare SettingWithCopyWarning e garantire che i valori vengano aggiornati
        X_tr.loc[X_tr['dataset'] == dataset, col] = X_tr.loc[X_tr['dataset'] == dataset, col].fillna(median)
        X_te.loc[X_te['dataset'] == dataset, col] = X_te.loc[X_te['dataset'] == dataset, col].fillna(median)
        
    for con in var_cat:
        #dati categorici: imputazione con moda
        mode = X_tr.loc[X_tr['dataset'] == dataset, con].mode()[0]
        
        X_tr.loc[X_tr['dataset'] == dataset, con] = X_tr.loc[X_tr['dataset'] == dataset, con].fillna(mode)
        X_te.loc[X_te['dataset'] == dataset, con] = X_te.loc[X_te['dataset'] == dataset, con].fillna(mode)
    

percentuale_mancanti_tr = X_tr.isna().sum() / X_tr.shape[0] * 100
print("\nPercentuale mancanti dopo imputazione:")
print(percentuale_mancanti_tr)

['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca']
Valori mancanti prima dell'imputazione:
age           0
sex           0
dataset       0
cp            0
trestbps     49
chol         25
fbs          69
restecg       2
thalch       46
exang        46
oldpeak      50
slope       250
ca          487
thal        381
dtype: int64

Percentuale mancanti dopo imputazione:
age         0.0
sex         0.0
dataset     0.0
cp          0.0
trestbps    0.0
chol        0.0
fbs         0.0
restecg     0.0
thalch      0.0
exang       0.0
oldpeak     0.0
slope       0.0
ca          0.0
thal        0.0
dtype: float64


In [4]:
#buttare le righe che sono troppo sparse
'''
X_tr = X_tr.loc[X_tr.isnull().sum(axis=1) < 0.7 * X_tr.shape[1]]
X_te = X_te.loc[X_te.isnull().sum(axis=1) < 0.7 * X_te.shape[1]]
'''
print(X_tr.shape)
print(X_te.shape)

thershold = X_tr.shape[1] / 2
X_tr = X_tr.dropna(thresh=thershold)
X_te = X_te.dropna(thresh=thershold)

print(X_tr.shape)
print(X_te.shape)



(736, 14)
(184, 14)
(736, 14)
(184, 14)


In [5]:
#imputazione
#rimpiazzo i nan con la media
from sklearn.impute import SimpleImputer
col_num = [col for col in X_tr.columns if col not in var_cat]
print( X_tr.isna().sum()/X_tr.shape[0])

imputer1 = SimpleImputer(strategy='median')

X_tr[col_num] = imputer1.fit_transform(X_tr[col_num]) #fa la sostituzione direttamente
X_te[col_num] = imputer1.fit_transform(X_te[col_num]) 

imputer2 = SimpleImputer(strategy='most_frequent') 

X_tr[var_cat] = imputer1.fit_transform(X_tr[var_cat])
X_te[var_cat] = imputer1.fit_transform(X_te[var_cat])




age         0.0
sex         0.0
dataset     0.0
cp          0.0
trestbps    0.0
chol        0.0
fbs         0.0
restecg     0.0
thalch      0.0
exang       0.0
oldpeak     0.0
slope       0.0
ca          0.0
thal        0.0
dtype: float64


ValueError: Cannot use median strategy with non-numeric data:
could not convert string to float: 'Male'

In [6]:
#Feature Selection
from sklearn.feature_selection import VarianceThreshold

print(X_tr.var())

fs = VarianceThreshold(threshold=0.3)
X_tr_fs = fs.fit_transform(X_tr)

fs.get_support()

age            87.088761
sex             0.171131
dataset         1.278793
cp              0.941193
trestbps      322.566193
chol        11713.004538
fbs             0.132949
restecg         0.396917
thalch        653.295209
exang           0.232222
oldpeak         1.028651
slope           0.262223
ca              0.364737
thal            0.241822
dtype: float64


array([ True, False,  True,  True,  True,  True, False,  True,  True,
       False,  True, False,  True, False])

In [7]:
from sklearn.decomposition import PCA

# 1. Inizializzazione
pca = PCA(n_components=0.8)

# 2. SUL TRAIN: Fit + Transform
# Impara dai dati di train e li trasforma
X_train_pca = pca.fit_transform(X_tr)

# 3. SUL TEST: Solo Transform
# Usa la "mappa" creata dal train per trasformare il test
X_test_pca = pca.transform(X_te)

# Verifica
print(f"Componenti estratte dal train: {pca.n_components_}")
print(f"Forma del test dopo PCA: {X_test_pca.shape}")

Componenti estratte dal train: 1
Forma del test dopo PCA: (184, 1)
